# 01 · 데이터 수집 (Phase 1 파일럿)

> **파일럿 대상**: 삼성전자 `005930` × `fiscal_year=2024` (esg_year=2025)  
> **목표**: DART API로 사업보고서 1건을 수집하고 II/IV/VI 섹션에서 ESG passage 후보를 뽑는다.  
> **가이드**: `docs/supplements/versions/v0.2/COLLECTION_데이터수집가이드.html`

## 이 노트북에서 하는 것
1. 환경 셋업 (uv 로컬 / pip Colab 병행)
2. `stock_code` → `corp_code` 매핑
3. `corp_code` → `rcept_no` 조회 (사업보고서 2024.12)
4. `rcept_no` → `document.xml` 다운로드 (zip)
5. XML 파싱 → II/IV/VI 섹션 추출
6. seed 30개로 ESG passage 후보 추출
7. viewer URL 출력 (수동 정합성 검증용)
8. `collection_log.csv`에 1행 기록

## 위반 금지 (이 노트북이 지키는 규칙)
- ✅ API key는 `.env`에서만 로드, 출력 ❌
- ✅ `stock_code` 기준 (회사명 join ❌)
- ✅ `esg_year = fiscal_year + 1`
- ✅ 실패 시 가짜 0 ❌, `collection_log`에 사유 기록
- ✅ document.xml이 표준 경로 (viewer 페이지 우회 ❌)
- ✅ MCP가 만든 passage는 corpus에 직접 투입 ❌ (Python 재현)

---
## Cell 1 · 환경 셋업

**로컬 (uv)**: 이 노트북 실행 전 터미널에서 `uv venv && uv pip install -e .`  
**Colab (pip)**: 아래 셀의 `!pip install ...` 주석을 해제하고 실행.

In [ ]:
# Colab 사용자만 주석 해제
# !pip install -r ../requirements.txt -q

import os
import json
import re
import time
import zipfile
from io import BytesIO
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
from dotenv import load_dotenv
from lxml import etree
from tqdm.auto import tqdm

# .env 로드 (로컬)
load_dotenv(dotenv_path=Path('..') / '.env')

API_KEY = os.getenv('OPENDART_API_KEY')

# Colab Secrets 사용 시:
# from google.colab import userdata
# API_KEY = userdata.get('OPENDART_API_KEY')

assert API_KEY, '❌ OPENDART_API_KEY가 .env에 없습니다. .env.example 참고해 .env 만들고 키 입력.'
print(f'✅ API key 로드 완료 (길이: {len(API_KEY)})  # 키 자체는 출력 ❌')

# 디렉토리 셋업 (프로젝트 루트 기준)
ROOT = Path('..').resolve()
DATA_RAW = ROOT / 'data' / 'raw'
DATA_INTERIM = ROOT / 'data' / 'interim'
DATA_CACHE = ROOT / 'data' / 'cache'
LOGS = ROOT / 'logs'

for d in [DATA_RAW, DATA_INTERIM, DATA_CACHE, LOGS]:
    d.mkdir(parents=True, exist_ok=True)

print(f'📁 출력 디렉토리 준비 완료')
print(f'   raw:     {DATA_RAW}')
print(f'   interim: {DATA_INTERIM}')
print(f'   logs:    {LOGS}')

---
## 파일럿 대상 설정

이번 파일럿은 **삼성전자 005930 × 2024**. company_master.csv에서도 같은 행을 확인.

In [ ]:
# 파일럿 변수
PILOT_STOCK_CODE = '005930'
PILOT_FISCAL_YEAR = 2024
PILOT_ESG_YEAR = PILOT_FISCAL_YEAR + 1

# company_master.csv 확인
master = pd.read_csv(ROOT / 'data' / 'company_master.csv', dtype={'stock_code': str})
print(f'📊 company_master: {len(master)}행 (예상 381)')
print(f'   unique stock_code: {master["stock_code"].nunique()}개 (예상 127)')
print(f'   fiscal_year: {sorted(master["fiscal_year"].unique())}')

# 파일럿 행 확인
pilot_row = master[(master['stock_code'] == PILOT_STOCK_CODE) & (master['fiscal_year'] == PILOT_FISCAL_YEAR)]
assert len(pilot_row) == 1, f'❌ 파일럿 행을 찾을 수 없음: {PILOT_STOCK_CODE} / {PILOT_FISCAL_YEAR}'
pilot_row.T

---
## Cell 2 · corp_code 매핑

OpenDART의 `corpCode.xml` 엔드포인트는 **모든 기업의 corp_code 사전**을 zip으로 내려줍니다. 한 번만 받아서 캐시.

**왜 이게 필요한가?** OpenDART의 다른 모든 API는 `corp_code`(8자리)를 키로 받습니다. `stock_code`(6자리)는 그 자체로는 못 씁니다.

In [ ]:
def fetch_corp_code_map(api_key: str, cache_path: Path) -> pd.DataFrame:
    """OpenDART corpCode.xml을 받아 stock_code ↔ corp_code 매핑을 DataFrame으로 반환.
    이미 받은 게 있으면 캐시에서 로드."""
    cache_df = cache_path / 'corp_code_map.parquet'
    if cache_df.exists():
        print(f'📦 캐시에서 로드: {cache_df.name}')
        return pd.read_parquet(cache_df)

    print('🌐 OpenDART corpCode.xml 다운로드 중...')
    res = requests.get(
        'https://opendart.fss.or.kr/api/corpCode.xml',
        params={'crtfc_key': api_key},
        timeout=60,
    )
    res.raise_for_status()

    with zipfile.ZipFile(BytesIO(res.content)) as zf:
        xml_bytes = zf.read(zf.namelist()[0])

    root = etree.fromstring(xml_bytes)
    rows = []
    for item in root.findall('list'):
        rows.append({
            'corp_code': item.findtext('corp_code', '').strip(),
            'corp_name': item.findtext('corp_name', '').strip(),
            'stock_code': item.findtext('stock_code', '').strip(),
        })
    df = pd.DataFrame(rows)
    df = df[df['stock_code'] != '']  # 상장사만 유지
    df.to_parquet(cache_df, index=False)
    print(f'✅ 매핑 {len(df):,}건 저장: {cache_df}')
    return df


corp_map = fetch_corp_code_map(API_KEY, DATA_CACHE)
print(f'\n전체 매핑: {len(corp_map):,}건')
corp_map.head()

In [ ]:
# 파일럿 stock_code의 corp_code 확인
lookup = corp_map[corp_map['stock_code'] == PILOT_STOCK_CODE]
assert len(lookup) >= 1, f'❌ {PILOT_STOCK_CODE}의 corp_code 매핑 실패'

PILOT_CORP_CODE = lookup.iloc[0]['corp_code']
PILOT_CORP_NAME = lookup.iloc[0]['corp_name']
print(f'✅ stock_code={PILOT_STOCK_CODE} → corp_code={PILOT_CORP_CODE} ({PILOT_CORP_NAME})')

---
## Cell 3 · rcept_no 조회

`list.json`으로 해당 기업의 **2025년 공시 목록**을 받고, `report_nm`에 `"사업보고서"` AND `"(2024.12)"` 포함된 행을 선택.

- `bgn_de=20250101`, `end_de=20251231` (사업보고서는 다음 연도 3월경 제출)
- `pblntf_detail_ty='A001'` (사업보고서)
- `last_reprt_at='Y'` (정정 시 최신만)

In [ ]:
def find_rcept_no(api_key: str, corp_code: str, fiscal_year: int):
    """corp_code와 fiscal_year로 사업보고서 rcept_no를 찾아 dict로 반환."""
    search_year = fiscal_year + 1  # 사업보고서는 다음 연도 제출
    params = {
        'crtfc_key': api_key,
        'corp_code': corp_code,
        'bgn_de': f'{search_year}0101',
        'end_de': f'{search_year}1231',
        'pblntf_detail_ty': 'A001',
        'last_reprt_at': 'Y',
        'page_count': '100',
    }
    res = requests.get(
        'https://opendart.fss.or.kr/api/list.json',
        params=params,
        timeout=20,
    )
    res.raise_for_status()
    payload = res.json()
    if payload.get('status') != '000':
        return {'status': 'FAIL_LIST', 'reason': payload.get('message', '')}

    rows = payload.get('list', [])
    target_marker = f'({fiscal_year}.12)'
    candidates = [
        r for r in rows
        if '사업보고서' in r.get('report_nm', '') and target_marker in r.get('report_nm', '')
    ]

    if not candidates:
        return {'status': 'FAIL_RCEPT', 'reason': f'사업보고서 ({fiscal_year}.12) 없음 (총 {len(rows)}건 중)'}

    chosen = candidates[0]
    return {
        'status': 'SUCCESS',
        'rcept_no': chosen['rcept_no'],
        'rcept_dt': chosen['rcept_dt'],
        'report_nm': chosen['report_nm'],
        'viewer_url': f'https://dart.fss.or.kr/dsaf001/main.do?rcpNo={chosen["rcept_no"]}',
    }


result = find_rcept_no(API_KEY, PILOT_CORP_CODE, PILOT_FISCAL_YEAR)
assert result['status'] == 'SUCCESS', f'❌ rcept_no 조회 실패: {result}'

PILOT_RCEPT_NO = result['rcept_no']
PILOT_VIEWER_URL = result['viewer_url']
print(f'✅ rcept_no={PILOT_RCEPT_NO}')
print(f'   report_nm={result["report_nm"]}')
print(f'   rcept_dt={result["rcept_dt"]}')
print(f'   viewer:   {PILOT_VIEWER_URL}')

---
## Cell 4 · document.xml 다운로드

`rcept_no`로 사업보고서 원문 zip을 받아 `data/raw/`에 저장. 이미 받은 zip은 재다운로드 ❌ (idempotent).

In [ ]:
def download_document(api_key: str, rcept_no: str, stock_code: str, fiscal_year: int):
    """document.xml을 받아 data/raw/에 저장하고 XML 텍스트 반환."""
    out_path = DATA_RAW / f'{stock_code}_{fiscal_year}_disclosure.zip'

    if out_path.exists() and out_path.stat().st_size > 1000:
        print(f'📦 캐시에서 로드: {out_path.name} ({out_path.stat().st_size / 1024:.1f} KB)')
        zip_bytes = out_path.read_bytes()
    else:
        print(f'🌐 다운로드 중: rcept_no={rcept_no}')
        res = requests.get(
            'https://opendart.fss.or.kr/api/document.xml',
            params={'crtfc_key': api_key, 'rcept_no': rcept_no},
            timeout=60,
        )
        res.raise_for_status()
        zip_bytes = res.content
        out_path.write_bytes(zip_bytes)
        print(f'✅ 저장: {out_path.name} ({len(zip_bytes) / 1024:.1f} KB)')

    with zipfile.ZipFile(BytesIO(zip_bytes)) as zf:
        xml_bytes = zf.read(zf.namelist()[0])
    xml_text = xml_bytes.decode('utf-8', errors='ignore')
    return xml_text, out_path


xml_text, zip_path = download_document(API_KEY, PILOT_RCEPT_NO, PILOT_STOCK_CODE, PILOT_FISCAL_YEAR)
print(f'\n📄 XML 길이: {len(xml_text):,}자 (예상 100K+)')
print(f'   앞 200자: {xml_text[:200]}')

# Sanity check
assert len(xml_text) > 10000, f'⚠️ XML이 너무 짧음: {len(xml_text)}자 — error 응답 가능성'

---
## Cell 5 · 섹션 추출 (II · IV · VI)

DART XML에서 다음 섹션을 추출:
- **II. 사업의 내용** — E 신호 多 (탄소·재생에너지·폐기물 등)
- **IV. 이사의 경영진단 및 분석의견** — S·G 신호 (안전·공급망·이사회 등)
- **VI. 이사회 등 회사의 기관에 관한 사항** — G 신호 多 (사외이사·감사위 등)

**주의** — 회사마다 헤딩 형식이 약간 다름 (`II.`, `Ⅱ.`, `2.`). 여러 패턴 OR로 매칭.

In [ ]:
def strip_tags(xml_text: str) -> str:
    """XML 태그 제거. lxml이 무거우면 정규식 fallback."""
    try:
        root = etree.fromstring(xml_text.encode('utf-8'), parser=etree.XMLParser(recover=True))
        text = ''.join(root.itertext())
    except Exception:
        text = re.sub(r'<[^>]+>', ' ', xml_text)
    text = re.sub(r'\s+', ' ', text)
    return text


def extract_sections(xml_text: str) -> dict:
    """II/IV/VI 섹션을 dict로 반환."""
    plain = strip_tags(xml_text)

    # 헤딩 패턴 — 여러 변형을 OR로
    patterns = {
        'II': r'(?:II|Ⅱ|2)\.\s*사업의\s*내용',
        'III': r'(?:III|Ⅲ|3)\.\s*재무에\s*관한\s*사항',
        'IV': r'(?:IV|Ⅳ|4)\.\s*이사의\s*경영진단(?:\s*및\s*분석의견)?',
        'V': r'(?:V|Ⅴ|5)\.\s*(?:회계)?감사인',
        'VI': r'(?:VI|Ⅵ|6)\.\s*이사회',
        'VII': r'(?:VII|Ⅶ|7)\.\s*주주',
    }

    # 각 헤딩의 시작 위치 찾기
    positions = {}
    for key, pat in patterns.items():
        m = re.search(pat, plain)
        if m:
            positions[key] = m.start()

    # 위치 순서로 정렬해 각 섹션의 끝을 결정
    sorted_keys = sorted(positions.keys(), key=lambda k: positions[k])
    sections = {}
    for i, key in enumerate(sorted_keys):
        start = positions[key]
        end = positions[sorted_keys[i + 1]] if i + 1 < len(sorted_keys) else len(plain)
        sections[key] = plain[start:end].strip()

    return sections


sections = extract_sections(xml_text)
print('📑 추출된 섹션:')
for key, body in sections.items():
    marker = '⭐' if key in ('II', 'IV', 'VI') else '  '
    print(f'   {marker} {key}: {len(body):,}자 (앞 80자: {body[:80]}...)')

# ESG 분석 대상 섹션만 추출
target_sections = {k: sections[k] for k in ('II', 'IV', 'VI') if k in sections}
assert 'II' in target_sections, '❌ II 섹션이 추출되지 않음 — 정규식 패턴 점검'

In [ ]:
# data/interim/에 섹션 저장
out_sections = DATA_INTERIM / f'{PILOT_STOCK_CODE}_{PILOT_FISCAL_YEAR}_sections.json'
with open(out_sections, 'w', encoding='utf-8') as f:
    json.dump(target_sections, f, ensure_ascii=False, indent=2)
print(f'✅ 섹션 저장: {out_sections.name}')
print(f'   총 길이: {sum(len(v) for v in target_sections.values()):,}자')

---
## Cell 6 · ESG passage 후보 추출

seed 30개(E·S·G × 10) 키워드를 기준으로 **문장 단위**로 후보 passage를 추출.

**주의** — 이 단계의 출력은 "후보"일 뿐. 최종 corpus는 **다음 단계(전처리)**에서 결정합니다.

In [ ]:
# seed_dictionary.csv 로드
seeds = pd.read_csv(ROOT / 'data' / 'seed_dictionary.csv')
print(f'🌱 seed: {len(seeds)}개')
print(seeds.groupby('dimension')['seed_term'].apply(list))

# E·S·G별 키워드 집합
seed_dict = {dim: set(group['seed_term']) for dim, group in seeds.groupby('dimension')}

In [ ]:
def extract_passages(section_text: str, seed_dict: dict, min_len: int = 20):
    """문장 단위로 분리 후 seed 키워드 매칭된 문장만 dict 리스트로 반환."""
    # 문장 분리 — 마침표·줄바꿈 기준 (간단 버전. 더 정교한 분리는 KSS 등 사용 가능)
    sentences = re.split(r'(?<=[.。?!])\s+|\n+', section_text)
    sentences = [s.strip() for s in sentences if len(s.strip()) >= min_len]

    rows = []
    for sent in sentences:
        matched = {}
        for dim, terms in seed_dict.items():
            hits = [t for t in terms if t in sent]
            if hits:
                matched[dim] = hits
        if matched:
            rows.append({
                'sentence': sent,
                'dimensions': '|'.join(matched.keys()),
                'matched_terms': '|'.join(t for terms in matched.values() for t in terms),
                'n_matches': sum(len(v) for v in matched.values()),
                'len': len(sent),
            })
    return pd.DataFrame(rows)


# 섹션별 passage 후보 추출
all_passages = []
for sec_key, sec_text in target_sections.items():
    df = extract_passages(sec_text, seed_dict)
    df['section'] = sec_key
    all_passages.append(df)

passages = pd.concat(all_passages, ignore_index=True) if all_passages else pd.DataFrame()
print(f'\n📝 추출된 passage 후보: {len(passages)}개')
print(f'   섹션별: {passages.groupby("section").size().to_dict()}')
print(f'   차원별: {passages["dimensions"].value_counts().to_dict() if len(passages) else {}}')

In [ ]:
# top 10 passage 미리보기
if len(passages):
    top = passages.nlargest(10, 'n_matches')[['section', 'dimensions', 'matched_terms', 'sentence']]
    # 문장은 짧게 잘라서 표시
    top['sentence'] = top['sentence'].str.slice(0, 120) + '...'
    display(top)

---
## Cell 7 · viewer URL 정합성 검증

**수동 검증 단계** — 위에서 추출한 passage 5개를 무작위로 골라 viewer URL에서 직접 확인. AI(MCP)가 만든 passage가 아니라 실제 공시 문장인지 확인.

**판정 기준** — 5개 중 4개 이상이 viewer 원문과 일치하면 ✅.

In [ ]:
print(f'🔗 viewer URL: {PILOT_VIEWER_URL}')
print('\n👉 위 URL을 브라우저에서 열어 다음 5개 문장이 원문에 있는지 확인하세요:\n')

if len(passages):
    sample = passages.sample(min(5, len(passages)), random_state=42)
    for i, (_, row) in enumerate(sample.iterrows(), 1):
        print(f'[{i}] 섹션 {row["section"]} · 차원 {row["dimensions"]} · 매치 {row["matched_terms"]}')
        print(f'    {row["sentence"][:200]}...\n')

print('체크: viewer URL에서 위 5개 문장 중 몇 개가 발견되나요?')
print('  - 4개 이상 일치  →  ✅ 추출 정상')
print('  - 3개 이하       →  ⚠️ 정규식·인코딩 점검 필요')

---
## Cell 8 · collection_log.csv 기록

각 firm-year의 수집 시도·결과를 한 행씩 누적. 실패해도 사유와 함께 기록 (가짜 0 ❌).

In [ ]:
def append_log(stock_code, fiscal_year, corp_code, rcept_no, status, reason='', n_passages=None):
    """collection_log.csv에 한 행 append."""
    log_path = LOGS / 'collection_log.csv'
    row = {
        'stock_code': stock_code,
        'fiscal_year': fiscal_year,
        'esg_year': fiscal_year + 1,
        'corp_code': corp_code or '',
        'rcept_no': rcept_no or '',
        'status': status,
        'reason': reason,
        'n_passages': n_passages if n_passages is not None else '',
        'timestamp': datetime.now().isoformat(timespec='seconds'),
    }
    df_row = pd.DataFrame([row])
    df_row.to_csv(
        log_path,
        mode='a',
        header=not log_path.exists(),
        index=False,
        encoding='utf-8',
    )
    return log_path


log_path = append_log(
    stock_code=PILOT_STOCK_CODE,
    fiscal_year=PILOT_FISCAL_YEAR,
    corp_code=PILOT_CORP_CODE,
    rcept_no=PILOT_RCEPT_NO,
    status='SUCCESS',
    reason='',
    n_passages=len(passages),
)
print(f'✅ collection_log 기록: {log_path}')
pd.read_csv(log_path).tail(5)

---
## 🎉 파일럿 완료 체크리스트

- [x] API key 안전 로드 (출력 ❌)
- [x] corp_code 매핑 성공: `005930` → `{PILOT_CORP_CODE}`
- [x] rcept_no 조회 성공: `{PILOT_RCEPT_NO}`
- [x] document.xml zip 저장: `data/raw/005930_2024_disclosure.zip`
- [x] II/IV/VI 섹션 추출 성공
- [x] ESG passage 후보 ≥ 30개 (실제 결과는 출력 참고)
- [x] viewer URL 정합성 확인 (수동 — 위에서 직접)
- [x] collection_log.csv에 1행 기록

## 다음 단계

1. **회고 회의** — 팀 4명이 각자 파일럿 결과 비교, Phase 2 표준 결정
2. **Phase 2 진입** — 동일 함수를 381 행에 반복 적용 (`src/collection.py`로 모듈화 + rate limit + checkpoint)
3. **다음 노트북** — `02_preprocessing.ipynb` (한국어 토큰화·불용어·TF-IDF 준비)

**참고 문서**
- `docs/supplements/versions/v0.2/COLLECTION_데이터수집가이드.html` — 전체 수집 가이드
- `reports/01_collection_README.md` — 단계별 결정 사항 (다음 작성 예정)
- `docs/professor_md/02_dart_data_collection.md` — 교수님 가이드 원본